# F1 — W3 Mar 14 Flash Spike at 14:00 UTC

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [ ]:
wm = WINDOWS_META['W3']
day = '2025-03-14'
mids = {}
for sym in [wm['front'], wm['back']]:
    fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
    assert fpath, f'Missing {sym} {day}'
    df = pd.read_parquet(fpath[0], columns=['ts_event','bid_px_00','ask_px_00'])
    df['ts_event'] = pd.to_datetime(df['ts_event'])
    mask = (df['ts_event'].dt.hour == 13) & (df['ts_event'].dt.minute >= 55) |            (df['ts_event'].dt.hour == 14) & (df['ts_event'].dt.minute <= 5)
    df = df[mask].set_index('ts_event')
    mids[sym] = ((df['bid_px_00']+df['ask_px_00'])/2).resample('1ms').last().ffill()
    del df; gc.collect()

cal = mids[wm['back']].subtract(mids[wm['front']]).dropna()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(cal.index, cal.values, lw=0.8, color='steelblue')
ax.axvline(pd.Timestamp('2025-03-14 14:00:00'), color='red', ls='--', lw=1.2, label='14:00 UTC')
# Annotate trade 27 (entry ~14:00:01, Short SL -$218.75)
ax.annotate('Trade 27\nShort SL\n−$218.75',
            xy=(pd.Timestamp('2025-03-14 14:00:01'), cal.reindex([pd.Timestamp('2025-03-14 14:00:01')], method='nearest').values[0]),
            xytext=(pd.Timestamp('2025-03-14 13:58:00'), cal.max()*0.9),
            fontsize=8, color='red', arrowprops=dict(arrowstyle='->', color='red'))
# Annotate trade 26 (entry ~13:45, running into spike, -$93.75)
ax.annotate('Trade 26\nShort SL\n−$93.75',
            xy=(pd.Timestamp('2025-03-14 14:00:00'), cal.reindex([pd.Timestamp('2025-03-14 14:00:00')], method='nearest').values[0]),
            xytext=(pd.Timestamp('2025-03-14 13:56:00'), cal.max()*0.7),
            fontsize=8, color='darkorange', arrowprops=dict(arrowstyle='->', color='darkorange'))
ax.set_xlabel('Time (UTC)'); ax.set_ylabel('Calendar Spread (pts)')
ax.set_title('W3 ESH5→ESM5: 14:00 UTC Flash Spike — 2025-03-14 (1ms resolution)')
ax.legend()
save_fig(fig, 'F1_w3_mar14_spike.png')
